"""Default script for training and validating a single CF model."""

In [0]:
%run ../../config/utils

In [0]:
from datetime import datetime
from dateutil import parser
import time
import os
import sys
sys.path.append('..')
sys.path.append('../..')


from pyspark import SparkContext
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.sql.functions import (
    when,
)
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.client import MlflowClient

from lib_cf.models import (
    predict_pf,
    fit_output_scaler,
    evaluate_cf_train,
)
from lib_cf.models import (
    scale_by_group,
    purchase_cycle_normalize,
    cat_size_normalize,
    cap_outliers,
)
from lib_cf.cf_io import (
    load_config,
    calculate_filepaths,
    save_scaler,
)

In [0]:
config_path = dbutils.widgets.get("config_path")
test = False if not dbutils.widgets.get("test") else True
rmse = False if not dbutils.widgets.get("rmse") else True
future = False if not dbutils.widgets.get("future") else True

In [0]:
# (1) ---- ARGUMENTS ---- #
CNF, CFG_PATH = load_config(config_path, test, future, rmse)
PARAMS = dict(list(CNF["shared"].items()) + list(CNF["train"].items()))
PATHS = CNF["paths"]
PARAMS, PATHS = calculate_filepaths(PARAMS, PATHS)

start_dt = parser.parse(PARAMS["start"])
end_dt = parser.parse(PARAMS["end"])
time_period = (end_dt - start_dt).days

In [0]:
# (3) ---- READ DATA ---- #
print("(1/5) starting data read...")
data = read_cf_tables(cf_matrix, PARAMS).select(
    "MBRSHP_SID", "CATEGORY_ID", PARAMS["inputtype"]
)

In [0]:
# (4) ---- Filter and Split ---- #
print("(2/5) starting data prep...")
if PARAMS["binary"]:
    data = data.withColumn(
        PARAMS["inputtype"],
        when(data[PARAMS["inputtype"]] > 0, 1).otherwise(0),
    )
if PARAMS["cat_norm"]:
    cat_dna = spark.table(globals()[f"fs_{PARAMS['category'].lower()}_cd_category_dna_full"])  
    data = purchase_cycle_normalize(
        PARAMS,
        cat_dna,
        data,
        time_period
    )
if PARAMS["member_norm"]:
    data = scale_by_group(data, PARAMS["inputtype"], "MBRSHP_SID", "mean")
if PARAMS["size_norm"]:
    item_brand = spark.table(silver_ad_hoc_master_item_with_brand)
    data = cat_size_normalize(
        PARAMS, item_brand, data
    )

data = cap_outliers(data, PARAMS["inputtype"])

n_examples = data.count()
print("training with: ", n_examples, "examples")

In [0]:
experiment_name = experiment_name_cf_model

mlflow.spark.autolog()

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

feature_dataset = mlflow.data.from_spark(data, name = 'cf_etl_dataset')

In [0]:
# (5) ---- Train Model ---- #
print("(3/5) starting train...")
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')
evaluating = "_evaluating_" if rmse else ''
registered_model_name = cf_model_catalog + f"_{PARAMS['category'].lower()}"
with mlflow.start_run(run_name=f'training{evaluating}_{PARAMS['category']}_{mark_datetime}') as run:
    mlflow.log_input(feature_dataset, context="source")
    als = ALS(
        maxIter=PARAMS["iter"],
        rank=PARAMS["rank"],  # depth of vector to learn
        regParam=PARAMS["lambda"],  # lambda
        alpha=PARAMS["alpha"],  # preference alpha
        userCol="MBRSHP_SID",
        itemCol="CATEGORY_ID",
        numUserBlocks=10,  # 'partitions' of members
        numItemBlocks=10,  # 'partitions' of categories
        checkpointInterval=10,  # interval on which to checkpoint the cache()
        nonnegative=True,
        ratingCol=PARAMS["inputtype"],
        implicitPrefs=True,
        coldStartStrategy="drop",
        # intermediateStorageLevel="MEMORY_AND_DISK",
        # finalStorageLevel="MEMORY_AND_DISK",
        seed=593367982098446717,
    )
    model = als.fit(data)

    # (6) ---- Evaluate Model ---- #

    if PARAMS["rmse"]:
        print("(4/5) evaluating...")
        slate = read_cf_tables(cf_slate, PARAMS)
        pdata = read_cf_tables(cf_matrix, PARAMS, past_flag=True)
        fdata = read_cf_tables(cf_matrix, PARAMS, future_flag=True)
        cat_lookup = read_cf_tables(cf_cat_lookup, PARAMS)
        evals = evaluate_cf_train(
            spark,
            model,
            pdata,
            slate,
            fdata,
            cat_lookup,
            eval_col=PARAMS["eval_col"],
            backtest=PARAMS["backtest"],
            level=PARAMS["category"],
            params=PARAMS,
        )
        top_cat = evals.pop('top_cat', None)
        evals.pop('backtest', None)
        mlflow.log_metrics(evals)
        mlflow.log_param("eval_top_cat", top_cat)
     
    else:
        backtest = None
        rmse = None
        mean_score = None
        count_under05 = None
        top_cat = None
        top_cat_ct = None
        hits_1 = None
        hits_05 = None
        overall = None
        personal = None
        lut = None

    if PARAMS["scaler"]:
        print("(6/5) training scaler")
        slate = read_cf_tables(cf_slate, PARAMS)
        pdata = read_cf_tables(cf_matrix, PARAMS, past_flag=True).select(
            "MBRSHP_SID", "CATEGORY_ID", PARAMS["eval_col"]
        )
        fdata = read_cf_tables(cf_matrix, PARAMS, future_flag=True).select(
            "MBRSHP_SID", "CATEGORY_ID", PARAMS["eval_col"]
        )
        cat_lookup = read_cf_tables(cf_cat_lookup, PARAMS)
        predictions = predict_pf(
            PARAMS,
            model,
            slate,
            pdata,
            fdata,
            cat_lookup=cat_lookup,
            col_of_interest=PARAMS["eval_col"],
        )
        scaler = fit_output_scaler(sc, predictions)
        print("(7/5) saving scaler")
        # save_scaler(sc, scaler, PATHS["slate"]) TODO: Revisar manejo

    mlflow.log_params({
        "userCol": "MBRSHP_SID",
        "itemCol": "CATEGORY_ID",
        "ratingCol": PARAMS["inputtype"],
        "maxIter": als.getMaxIter(),
        "rank": als.getRank(),
        "regParam": als.getRegParam(),
        "alpha": als.getAlpha(),
        "implicitPrefs": als.getImplicitPrefs(),
        "nonnegative": als.getNonnegative(),
        "coldStartStrategy": als.getColdStartStrategy(),
        "numUserBlocks": als.getNumUserBlocks(),
        "numItemBlocks": als.getNumItemBlocks(),
    })

    mlflow.log_artifact(config_path)

    example_input_spark = (
        data.select("MBRSHP_SID", "CATEGORY_ID")
             .dropDuplicates()
             .limit(5)
    )
    example_output_spark = model.transform(example_input_spark).select("prediction")

    example_input_pd  = example_input_spark.toPandas()
    example_output_pd = example_output_spark.toPandas()

    signature = infer_signature(example_input_pd, example_output_pd)

    model_info = mlflow.spark.log_model(
            spark_model = model,
            artifact_path = "model",
            signature = signature,
            input_example=example_input_pd,
            registered_model_name = registered_model_name
        )

In [0]:
client = MlflowClient()
run_id = model_info.run_id

def find_version_by_run(model_name, run_id, max_wait_s=60):
    for _ in range(max_wait_s):
        for mv in client.search_model_versions(f"name = '{model_name}'"):
            if getattr(mv, "run_id", None) == run_id or str(mv.source).endswith("/model"):
                return int(mv.version)
        time.sleep(1)
    return None

version = getattr(model_info, "registered_model_version", None)
if version is None:
    version = find_version_by_run(registered_model_name, run_id)

if version is None:
    raise RuntimeError(f"Registered version not visible yet for {registered_model_name}")

if rmse:
    client.set_registered_model_alias(registered_model_name, "future", int(version))
else:
    client.set_registered_model_alias(registered_model_name, "champion", int(version))

client.set_model_version_tag(registered_model_name, str(version), 'start_date', start_dt.strftime('%Y-%m-%d'))
client.set_model_version_tag(registered_model_name, str(version), 'end_date', end_dt.strftime('%Y-%m-%d'))